# 8.01 InMemorySaver — 프로세스 내 체크포인터

`langgraph.checkpoint.memory.InMemorySaver` 는 LangGraph 본체에 포함된 **프로세스 내(in-memory)** 체크포인터다.
그래프 실행 중 각 super-step 의 상태를 파이썬 객체로 그대로 보관하므로 별도 서비스 없이 **thread_id 기반 대화 복원 · time travel** 을 바로 체험할 수 있다.

## 학습 목표

- `StateGraph.compile(checkpointer=InMemorySaver())` 로 체크포인터 붙이기
- `{"configurable": {"thread_id": "..."}}` 로 같은 대화에 이어 호출
- `graph.get_state(config)` / `graph.get_state_history(config)` 로 체크포인트 조회
- 특정 `checkpoint_id` 로 **과거 상태에서 재실행** (time travel)
- `create_agent(..., checkpointer=...)` 과의 동일 패턴

## 언제 쓰나

- 단위 테스트 · 데모 · 노트북 학습
- 짧은 세션만 유지하면 되는 CLI 툴 / REPL
- 프로덕션이 아니라 **checkpoint API 자체를 먼저 익히고 싶을 때**

⚠️ 프로세스가 종료되면 상태가 사라진다. 영속이 필요하면 `02_sqlite` / `03_postgres` / `04_cosmosdb` 로 넘어간다.


## 8.01.1 환경 설정 · 서비스 연결

`InMemorySaver` 는 langgraph 코어에 포함되므로 별도 설치가 없다. `OPENAI_API_KEY` 만 확인한다.

In [ ]:
# !pip install -U langgraph langchain langchain-openai python-dotenv

import os
from dotenv import load_dotenv
load_dotenv(override=True)

# 이 노트북은 OPENAI_API_KEY(에이전트 예제용) + InMemorySaver probe 만 필요
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 필요"

from langgraph.checkpoint.memory import InMemorySaver
saver = InMemorySaver()
print("InMemorySaver ready:", type(saver).__name__)


## 8.01.2 기본 사용법 — `StateGraph.compile(checkpointer=...)`

가장 작은 그래프로 checkpoint 동작을 관찰한다. 상태에 `messages` 와 누적 카운터 `turn` 을 둔다.

In [ ]:
import operator
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages: Annotated[list, add_messages]
    turn: Annotated[int, operator.add]   # 호출마다 1 씩 누적 (reducer 사용 → 체크포인터 효과가 눈에 보임)

def respond(state: ChatState) -> ChatState:
    # 마지막 사용자 메시지를 그대로 echo + turn reducer 로 +1
    last = state["messages"][-1].content if state["messages"] else "(empty)"
    next_turn = state.get("turn", 0) + 1
    return {
        "messages": [{"role": "assistant", "content": f"echo[{next_turn}]: {last}"}],
        "turn": 1,   # reducer 가 누적
    }

builder = StateGraph(ChatState)
builder.add_node("respond", respond)
builder.add_edge(START, "respond")
builder.add_edge("respond", END)

graph = builder.compile(checkpointer=saver)
print("compiled with checkpointer:", graph.checkpointer.__class__.__name__)


## 8.01.3 thread_id 스코프와 복원

체크포인터가 붙으면 `invoke` 호출 시 `config={"configurable": {"thread_id": "..."}}` 를 반드시 전달해야 한다.
같은 `thread_id` 로 다시 호출하면 **이전 상태에 이어서** 실행되고, 새 `thread_id` 로는 **독립된 대화** 가 된다.

In [ ]:
cfg_a = {"configurable": {"thread_id": "user-A"}}
cfg_b = {"configurable": {"thread_id": "user-B"}}

# user-A: 세 번 호출 — reducer(operator.add) 로 turn 이 누적된다
for msg in ["안녕", "오늘 날씨 어때?", "고마워"]:
    out_a = graph.invoke({"messages": [{"role": "user", "content": msg}]}, cfg_a)
print("A turn:", out_a["turn"], "| A last:", out_a["messages"][-1].content)

# user-B: 독립 세션 → turn 은 1 에서 시작
out_b = graph.invoke({"messages": [{"role": "user", "content": "처음입니다"}]}, cfg_b)
print("B turn:", out_b["turn"], "| B last:", out_b["messages"][-1].content)

# 다시 user-A 로 호출하면 이전 turn 을 이어받아 +1
out_a2 = graph.invoke({"messages": [{"role": "user", "content": "또 왔어"}]}, cfg_a)
print("A resume turn:", out_a2["turn"])


## 8.01.4 `get_state` · `get_state_history`

`graph.get_state(config)` 는 해당 thread 의 **최신 체크포인트** 를 반환한다. `graph.get_state_history(config)` 는 과거 체크포인트를 **최신 → 오래된 순** 으로 내어주는 iterator 다.

각 `StateSnapshot` 은 `values`, `next`, `config`, `metadata`, `parent_config` 를 가진다. `config["configurable"]["checkpoint_id"]` 를 저장해두면 그 시점으로 되돌아갈 수 있다.

In [ ]:
snap = graph.get_state(cfg_a)
print("latest values.turn =", snap.values["turn"])
print("latest checkpoint_id =", snap.config["configurable"]["checkpoint_id"][:16], "...")

history = list(graph.get_state_history(cfg_a))
print(f"\nthread user-A 의 체크포인트 수: {len(history)}")
for i, h in enumerate(history[:6]):
    print(f"  [{i}] turn={h.values.get('turn')} next={h.next} id={h.config['configurable']['checkpoint_id'][:12]}")


## 8.01.5 Time travel — 과거 체크포인트에서 재실행

원하는 체크포인트의 `config` 를 `invoke` 에 그대로 넘기면, 그 시점 상태로 **분기(branch)** 한다.
새 분기의 체크포인트는 기존 체크포인트를 `parent_config` 로 가리키므로 대화가 트리 구조가 된다.

In [ ]:
# turn == 1 인 시점 (첫 응답 직후 · 가장 오래된 상태) 을 찾는다
target = next(h for h in reversed(history) if h.values.get("turn") == 1 and h.next == ())
branch_cfg = target.config  # 이 checkpoint_id 로 fork

print("branch 시작 시점 turn =", target.values["turn"])

# 이 시점에서 다른 질문으로 분기 → 분기 쪽 turn 은 2 (1 에서 +1)
branched = graph.invoke(
    {"messages": [{"role": "user", "content": "다른 길로 가볼래"}]},
    branch_cfg,
)
print("branched turn =", branched["turn"])
print("branched last =", branched["messages"][-1].content)

# fork 이후 get_state(cfg_a) 는 가장 최근 체크포인트(=branched) 를 돌려준다.
# 원본 타임라인(turn 4 까지 쌓인 히스토리) 은 체크포인트 트리에 남아있다 — history 를 다시 뽑아보면 확인 가능.
print("\nlatest state on thread (branch 후 최신) turn =", graph.get_state(cfg_a).values["turn"])
print("history 길이 (원본 + branch 모두 포함) =", len(list(graph.get_state_history(cfg_a))))


## 8.01.6 `create_agent` 와의 결합

LangChain v1 의 `create_agent` 는 내부적으로 LangGraph 를 쓰므로, 동일한 `checkpointer=` 파라미터를 받는다.
멀티턴 에이전트에 메모리를 주고 싶다면 아래 패턴이 가장 단순하다.

In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain.tools import tool

@tool
def remember(fact: str) -> str:
    """사용자가 말한 사실을 로그로 남긴다."""
    return f"[recorded] {fact}"

agent = create_agent(
    model=ChatOpenAI(model="gpt-4.1-mini", temperature=0),
    tools=[remember],
    checkpointer=InMemorySaver(),   # 에이전트 전용 새 saver
    system_prompt="너는 간단한 메모장 에이전트야. 필요하면 remember 도구를 쓰세요.",
)

cfg = {"configurable": {"thread_id": "agent-demo-1"}}
r1 = agent.invoke({"messages": [{"role": "user", "content": "내 이름은 민지야"}]}, cfg)
print("1:", r1["messages"][-1].content[:120])

r2 = agent.invoke({"messages": [{"role": "user", "content": "내 이름이 뭐였지?"}]}, cfg)
print("2:", r2["messages"][-1].content[:120])


## 8.01.7 Subgraph 체크포인터 스코프

LangGraph 1.x 에서 subgraph 의 체크포인팅은 **상위 그래프의 checkpointer** 를 상속하는 게 기본값이다.
subgraph 만 독립적으로 체크포인팅하고 싶으면 subgraph 컴파일 시에도 `checkpointer=` 를 주거나, 상위에서 `checkpointer=False` 로 끊는다.
자세한 규칙은 `docs/skills/langgraph-persistence.md` 로 이어진다.

In [ ]:
# subgraph 도 동일 saver 를 공유하는 기본 케이스
inner = StateGraph(ChatState)
inner.add_node("respond", respond)
inner.add_edge(START, "respond")
inner.add_edge("respond", END)
inner_graph = inner.compile()  # checkpointer 미지정 → 상위 설정 상속

outer = StateGraph(ChatState)
outer.add_node("inner", inner_graph)
outer.add_edge(START, "inner")
outer.add_edge("inner", END)
outer_graph = outer.compile(checkpointer=InMemorySaver())

cfg_s = {"configurable": {"thread_id": "sub-1"}}
outer_graph.invoke({"messages": [{"role": "user", "content": "hi"}]}, cfg_s)
outer_graph.invoke({"messages": [{"role": "user", "content": "again"}]}, cfg_s)
print("outer history 길이:", len(list(outer_graph.get_state_history(cfg_s))))
print("outer turn:", outer_graph.get_state(cfg_s).values["turn"])


## 체크리스트

- [ ] `compile(checkpointer=...)` 후 모든 invoke 에 `thread_id` 를 반드시 전달
- [ ] 동일 `thread_id` → 상태 이어받음, 새 `thread_id` → 독립 대화
- [ ] `get_state_history` 는 최신 → 오래된 순 iterator (list 변환해서 쓰기)
- [ ] time travel 은 **새 분기** 를 만든다 — 원본 타임라인은 보존됨
- [ ] 프로세스 종료 시 사라진다 → 영속이 필요하면 Sqlite/Postgres/Cosmos

## 다음

- `02_sqlite.ipynb` — 파일 기반 영속 체크포인트
- `03_postgres.ipynb` — 멀티 프로세스 · 확장 가능한 프로덕션 백엔드
- `09_stores/` — **스레드 간** 공유 메모리 (`BaseStore`)
- `docs/skills/langgraph-persistence.md` — subgraph 스코프 · `checkpoint_ns` 내부 구조

## 참고

- LangGraph 체크포인터: https://langchain-ai.github.io/langgraph/how-tos/persistence/
- `InMemorySaver` 는 `langgraph.checkpoint.memory` (코어 내장)
